### Restricted-data analysis not included

This block used participant-level baseline clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level baseline event and clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from pathlib import Path

# ======================================================
# INPUT
# ======================================================
INPUT_FILE = "figs1_final/clinical_oncoponct_four_cohorts_baseline_q70_with_modality/baseline_clinical_oncoponct_group_summary.csv"

OUT_DIR = Path("./extended_table1_latex_final")
OUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_TEX = OUT_DIR / "Extended_Table1.tex"


# ======================================================
# LOAD + SPLIT GROUPS
# ======================================================
def build_group_dfs(path):

    df = pd.read_csv(path)

    mapping = {
        "CMCS UNI-modality cohort": "uCMCS",
        "CMCS multimodality cohort": "mCMCS",
        "Shougang UNI-modality cohort": "uSG",
        "Shougang multimodality cohort": "mSG",
    }

    df["group"] = df["group"].map(mapping)

    group_dfs = {
        g: df[df["group"] == g].copy()
        for g in ["uCMCS", "mCMCS", "uSG", "mSG"]
    }

    return group_dfs


# ======================================================
# STAT FORMATTERS
# ======================================================
def median_iqr(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{np.median(x):.1f} ({np.percentile(x,25):.1f}–{np.percentile(x,75):.1f})"


def pct(x):
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().sum() == 0:
        return "NA"
    return f"{(x==1).mean()*100:.1f}\\%"


# ======================================================
# BUILD TABLE 1
# ======================================================
def build_table1(group_dfs):

    rows = []

    for name, df in group_dfs.items():

        row = {}

        # =========================
        # BASIC
        # =========================
        row["Dataset"] = name
        row["N"] = len(df)

        # =========================
        # AGE
        # =========================
        row["Age"] = median_iqr(df["ageEntry"])

        # =========================
        # MODALITIES
        # =========================
        row["SNP"] = (df["modality_SNP"] == 1).sum() if "modality_SNP" in df.columns else 0
        row["Proteomics"] = (df["modality_Soma"] == 1).sum() if "modality_Soma" in df.columns else 0
        row["ECG"] = (df["modality_ECG"] == 1).sum() if "modality_ECG" in df.columns else 0
        row["Fundus"] = (df["modality_Fundus"] == 1).sum() if "modality_Fundus" in df.columns else 0

        # =========================
        # OUTCOMES
        # =========================
        row["CHD (%)"] = pct(df["fCHD"])
        row["Stroke (%)"] = pct(df["fistroke"])
        row["ASCVD (%)"] = pct(df["fICVD"])

        # =========================
        # RISK FACTORS
        # =========================
        row["Male (%)"] = pct(df["sexStatus"])
        row["Hypertension (%)"] = pct(df["hypertensionStatus"])
        row["Diabetes (%)"] = pct(df["diabeteStatus"])
        row["Dyslipidemia (%)"] = pct(df["dyslipidemiaStatus"])

        # =========================
        # CONTINUOUS (TRUE MEDIAN + IQR)
        # =========================
        continuous = [
            "SBP", "DBP",
            "TC", "TG",
            "LDLC", "HDLC",
            "BMI", "Waist",
            "FBG"
        ]

        for c in continuous:
            if c in df.columns:
                row[c] = median_iqr(df[c])
            else:
                row[c] = "NA"

        rows.append(row)

    return pd.DataFrame(rows).set_index("Dataset")


# ======================================================
# EXPORT LATEX (OVERLEAF READY)
# ======================================================
def export_latex(df, out_path):

    df = df.copy()

    def safe(x):
        if isinstance(x, (float, int, np.floating)):
            return f"{x:.1f}"
        return str(x)

    df = df.applymap(safe)

    latex = df.to_latex(
        index=True,
        escape=False,
        column_format="l" + "c"*df.shape[1]
    )

    latex = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\caption{Extended Table 1 | Multimodal Clinical Characteristics}\n"
        "\\small\n"
        + latex +
        "\\end{table}\n"
    )

    latex = latex.replace("%", "\\%")

    with open(out_path, "w") as f:
        f.write(latex)

    print("\n==============================")
    print("✔ FINAL TABLE SAVED:")
    print(out_path)
    print("==============================")


# ======================================================
# MAIN (ONE CLICK RUN)
# ======================================================
def main():

    print("Loading CSV ...")

    group_dfs = build_group_dfs(INPUT_FILE)

    print("Groups:", list(group_dfs.keys()))

    print("\nBuilding Table 1 ...")

    table1 = build_table1(group_dfs)

    print(table1)

    table1.to_csv(OUT_DIR / "Extended_Table1.csv")

    export_latex(table1, OUT_TEX)

    print("\nDONE ✔")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from pathlib import Path

# ======================================================
# INPUT
# ======================================================
INPUT_FILE = "figs1_final/clinical_oncoprint_four_cohorts_baseline_q70_with_modality/baseline_clinical_oncoprint_group_summary.csv"

OUT_DIR = Path("./extended_table1_latex")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# ======================================================
# COLUMN ORDER
# ======================================================
COL_MAP = {
    "CMCS UNI-modality cohort": "uCMCS",
    "CMCS multimodality cohort": "mCMCS",
    "Shougang UNI-modality cohort": "uSG",
    "Shougang multimodality cohort": "mSG",
}

COL_ORDER = ["uCMCS", "mCMCS", "uSG", "mSG"]


# ======================================================
# LOAD
# ======================================================
def load_summary(path):
    df = pd.read_csv(path)
    df["group"] = df["group"].map(COL_MAP)
    df = df.set_index("group").loc[COL_ORDER].reset_index()
    return df


# ======================================================

# ======================================================
def build_matrix(df):

    mat = {}

    mat["Age"] = df["age_median"].values

    mat["SNP"] = df["SNP_n_present"].values
    mat["Proteomics"] = df["Soma_n_present"].values
    mat["ECG"] = df["ECG_n_present"].values
    mat["Fundus"] = df["Fundus_n_present"].values

    mat["CHD (%)"] = df["CHD_rate"].values * 100
    mat["Stroke (%)"] = df["Ischemic stroke_rate"].values * 100
    mat["ASCVD (%)"] = df["ASCVD_rate"].values * 100

    mat["Male (%)"] = df["Sex_rate"].values * 100
    mat["Hypertension (%)"] = df["Hypertension_rate"].values * 100
    mat["Diabetes (%)"] = df["Diabetes_rate"].values * 100
    mat["Dyslipidemia (%)"] = df["Dyslipidemia_rate"].values * 100

    continuous_cols = [
        "SBP", "DBP",
        "TC", "TG",
        "LDL-C", "HDL-C",
        "BMI", "Waist",
        "FBG"
    ]

    for c in continuous_cols:
        qcol = f"{c}_Q70"
        if qcol in df.columns:
            mat[c] = df[qcol].values
        else:
            mat[c] = np.nan

    return pd.DataFrame(mat, index=COL_ORDER).T


# ======================================================
# 🔥 NEW: EXPORT LaTeX (THIS IS THE KEY CHANGE)
# ======================================================
def export_latex(mat, out_path):

    # transpose back to table format (rows = variables, cols = cohorts)
    df = mat.copy()

    # format numbers
    df = df.applymap(lambda x: f"{x:.1f}" if isinstance(x, (int, float, np.floating)) and not np.isnan(x) else str(x))

    latex = df.to_latex(
        index=True,
        escape=False,
        column_format="lcccc"
    )

    latex = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\caption{Extended Table 1 | Multimodal Clinical Characteristics}\n"
        "\\small\n"
        + latex +
        "\\end{table}\n"
    )

    with open(out_path, "w") as f:
        f.write(latex)

    print("\n==============================")
    print("LaTeX saved:")
    print(out_path)
    print("==============================")


# ======================================================
# MAIN
# ======================================================
def main():

    print("Loading summary ...")
    df = load_summary(INPUT_FILE)

    print("Shape:", df.shape)
    print(df.head())

    print("\nBuilding matrix ...")
    mat = build_matrix(df)

    print(mat)

    mat.to_csv(OUT_DIR / "Extended_Table1_matrix.csv")

    # 🔥 CHANGE HERE: heatmap → latex
    export_latex(mat, OUT_DIR / "Extended_Table1.tex")

    print("\nDONE ✔")


if __name__ == "__main__":
    main()

### Restricted-data analysis not included

This block used local shell listing. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from pathlib import Path

# ======================================================
# INPUT / OUTPUT
# ======================================================
INPUT_FILE = Path("figs1_final/clinical_oncoprint_four_cohorts_baseline_q70_with_modality/baseline_clinical_oncoprint_group_summary.csv")

OUT_DIR = Path("./extended_table1_latex_final")
OUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_TEX = OUT_DIR / "Extended_Table1.tex"


# ======================================================
# COLUMN MAP
# ======================================================
COL_MAP = {
    "CMCS UNI-modality cohort": "uCMCS",
    "CMCS multimodality cohort": "mCMCS",
    "Shougang UNI-modality cohort": "uSG",
    "Shougang multimodality cohort": "mSG",
}

COL_ORDER = ["uCMCS", "mCMCS", "uSG", "mSG"]


# ======================================================
# LOAD DATA
# ======================================================
def load_data(path):
    df = pd.read_csv(path)

    df["group"] = df["group"].map(COL_MAP)

    df = df.set_index("group").loc[COL_ORDER].reset_index()

    return df


# ======================================================
# FORMATTERS
# ======================================================
def fmt_num(x):
    try:
        return f"{float(x):.1f}"
    except:
        return "NA"


def fmt_pct(x):
    try:
        return f"{float(x)*100:.1f}"
    except:
        return "NA"


def fmt_count(x):
    try:
        return f"{int(x)}"
    except:
        return "NA"


# ======================================================
# BUILD TABLE
# ======================================================
def build_table(df):

    table = pd.DataFrame(index=COL_ORDER)

    # =========================
    # CORE DEMOGRAPHICS
    # =========================
    table["Age"] = df["age_median"].values

    # =========================
    # MODALITIES
    # =========================
    table["SNP"] = df["SNP_n_present"].values
    table["Proteomics"] = df["Soma_n_present"].values
    table["ECG"] = df["ECG_n_present"].values
    table["Fundus"] = df["Fundus_n_present"].values

    # =========================
    # OUTCOMES
    # =========================
    table["CHD (%)"] = df["CHD_rate"].values * 100
    table["Stroke (%)"] = df["Ischemic stroke_rate"].values * 100
    table["ASCVD (%)"] = df["ASCVD_rate"].values * 100

    # =========================
    # RISK FACTORS
    # =========================
    table["Male (%)"] = df["Sex_rate"].values * 100
    table["Hypertension (%)"] = df["Hypertension_rate"].values * 100
    table["Diabetes (%)"] = df["Diabetes_rate"].values * 100
    table["Dyslipidemia (%)"] = df["Dyslipidemia_rate"].values * 100

    # =========================
    # CONTINUOUS BIOMARKERS (Q70 fallback)
    # =========================
    continuous_cols = [
        "SBP_Q70",
        "DBP_Q70",
        "TC_Q70",
        "TG_Q70",
        "LDL-C_Q70",
        "HDL-C_Q70",
        "BMI_Q70",
        "Waist_Q70",
        "FBG_Q70"
    ]

    for c in continuous_cols:
        if c in df.columns:
            table[c.replace("_Q70", "")] = df[c]
        else:
            table[c.replace("_Q70", "")] = np.nan

    return table


def export_latex(df, out_path):

    df = df.copy()
    df = df.reset_index()

    n_cols = df.shape[1]
    col_format = "l" + "c"*n_cols

    latex = df.to_latex(
        index=False,
        escape=False,
        column_format=col_format
    )

    latex = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\caption{Extended Table 1 | Multimodal Clinical Characteristics}\n"
        "\\small\n"
        + latex +
        "\\end{table}\n"
    )

    with open(out_path, "w") as f:
        f.write(latex)
# ======================================================
# MAIN PIPELINE
# ======================================================
def main():

    print("Loading data ...")
    df = load_data(INPUT_FILE)

    print("Shape:", df.shape)
    print(df.head())

    print("\nBuilding table ...")
    table = build_table(df)

    print(table)

    table.to_csv(OUT_DIR / "Extended_Table1.csv")

    export_latex(table, OUT_TEX)

    print("\nDONE ✔")


if __name__ == "__main__":
    main()